In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision transformers matplotlib numpy pillow -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for package in ["torch", "torchvision", "transformers", "matplotlib", "numpy", "pillow"]:
    _install(package)

In [ ]:
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import TimesformerConfig, TimesformerForVideoClassification

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# TimeSformer 代码实战：学习实现 vs 工程实现

基于论文 *Is Space-Time Attention All You Need for Video Understanding?*（Bertasius et al., 2021），
用 **合成视频动作分类** 任务演示 TimeSformer 的核心思想：**把视频 token 的时间关系和空间关系分开建模**。

本 Notebook 保留两条并行路径，并且使用 **同一份数据、同一项任务、同一组评价指标**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 Divided Space-Time Attention 的内部计算 | 掌握 Hugging Face `TimesformerForVideoClassification` 的工程化用法 |
| 实现方式 | **L1：从零手写 tiny TimeSformer 并训练** | **E1：成熟库封装 + tiny config 训练** |
| 代码量 | 较多，显式展示 patch / temporal / spatial / block | 较少，高层 API 直接管理模块组装 |
| 适合场景 | 面试准备、原理拆解、研究型改动 | 快速验证、工程实验、参数管理 |

> 两条路径不是两套无关代码，而是同一套 TimeSformer 思想在不同抽象层级上的表达。

## Section i：论文背景、任务定义与范围

### 1. 论文与任务
- **论文**：*Is Space-Time Attention All You Need for Video Understanding?*
- **核心任务**：视频分类 / action recognition
- **核心创新**：把视频 Transformer 的注意力从联合时空注意力拆成 **Temporal Attention + Spatial Attention**，即论文中的 **Divided Space-Time Attention**。

### 2. 为什么这份 Notebook 选用合成视频分类
- 不依赖真实视频下载，打开即可运行。
- 可以显式构造“水平移动”和“垂直移动”两种模式，确保任务真的需要时序信息。
- tiny 配置可以在 CPU / 免费 Colab 上稳定训练，适合作为教学版 TimeSformer。

### 3. 本 Notebook 覆盖与不覆盖的边界
**覆盖：**
- patch embedding
- temporal self-attention
- spatial self-attention
- divided space-time block
- tiny TimeSformer 训练与推理
- Hugging Face 工程实现与对照

**不覆盖：**
- Kinetics-400 大规模预训练
- 长视频采样策略、稀疏采样、分布式训练
- 论文中全部 5 种变体的完整可运行复现（这里只把其余变体作为解释对象）

### 4. 路径决策
- **learning_path_depth = L1**：合成数据 + tiny 配置可以稳定训练完整模型。
- **engineering_path_form = E1**：Hugging Face `transformers` 已提供 TimeSformer 官方实现。
- **engineering_action = train tiny config**：为了和学习路径共用同一任务与同一数据，这里采用从 config 构建小模型并训练，而不是直接使用 Kinetics 预训练权重做异任务推理。  

## Section ii：最小必要理论

### 1. 从图像到视频 token
设输入视频为 $x \in \mathbb{R}^{T \times C \times H \times W}$，每帧切成 $N=(H/P)\times(W/P)$ 个 patch，则总 token 数为 $T\times N$。加入分类 token 后：

$$z_0 = [x_{cls}; x_{1,1}; \dots; x_{T,N}] + E_{pos}$$

### 2. TimeSformer 的核心不是“联合注意力”，而是“拆分注意力”
对每个 block：

$$x' = x + \mathrm{TemporalAttn}(\mathrm{LN}(x))$$
$$x'' = x' + \mathrm{SpatialAttn}(\mathrm{LN}(x'))$$
$$x_{out} = x'' + \mathrm{MLP}(\mathrm{LN}(x''))$$

- **Temporal Attention**：固定空间位置，跨帧做注意力。
- **Spatial Attention**：固定帧，在帧内所有 patch 之间做注意力。

### 3. 复杂度直觉
- Joint Space-Time Attention：每个 token 要看全部 $T \times N$ 个 token。
- Divided Space-Time Attention：先看 $T$，再看 $N$，计算更稳定，也更容易扩展到长视频。

### 4. 组件映射表（Mandatory）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Patch Embedding | `VideoPatchEmbedding` 手写 | `TimesformerEmbeddings` 内部 patch projection | Runnable |
| CLS Token | 显式 `nn.Parameter` | 模型内部 classification token | Runnable |
| Positional Embedding | 显式 `pos_embed` | 配置驱动的位置编码 | Runnable |
| Temporal Self-Attention | `TemporalSelfAttention` 手写 | TimeSformer block 内部 temporal attention | Runnable |
| Spatial Self-Attention | `SpatialSelfAttention` 手写 | TimeSformer block 内部 spatial attention | Runnable |
| Divided Space-Time Block | `DividedSTBlock` | `TimesformerForVideoClassification` 内部 encoder layers | Runnable |
| 分类头 | `nn.Linear(d_model, num_classes)` | `classifier` head | Runnable |
| Space-only Attention | 只做概念说明 | 可通过改注意力设计实现 | Explain-only |
| Joint Space-Time Attention | 只做复杂度对比 | 论文中的对照方案 | Explain-only |
| Local-Global / Axial 变体 | 只做比较说明 | 相关 Video Transformer 变体 | Explain-only |

## Section 1：数据准备

这里构造一个最小但有效的 **toy video classification** 数据集：
- **类别 0**：亮块沿水平方向移动
- **类别 1**：亮块沿垂直方向移动

为什么它适合 TimeSformer？
- 单帧外观非常相似，仅看某一帧并不总能区分方向。
- 真正的判别信息来自 **跨帧变化**。
- 这正好对应 TimeSformer 想解决的“时间 + 空间联合建模”问题。

In [ ]:
IMAGE_SIZE = 32
PATCH_SIZE = 8
NUM_FRAMES = 4
NUM_CLASSES = 2
MOTION_SIZE = 6

CLASS_NAMES = ['horizontal', 'vertical']

class SyntheticMotionVideoDataset(Dataset):
    def __init__(self, num_samples, image_size=32, num_frames=4, motion_size=6, noise_std=0.08):
        self.videos = []
        self.labels = []
        for idx in range(num_samples):
            label = idx % 2
            video = self._make_video(label, image_size, num_frames, motion_size, noise_std)
            self.videos.append(video)
            self.labels.append(label)
        self.videos = torch.stack(self.videos).float()   # (B, T, C, H, W)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def _make_video(self, label, image_size, num_frames, motion_size, noise_std):
        video = torch.randn(num_frames, 3, image_size, image_size) * noise_std
        base = 4
        step = 5
        fixed_row = image_size // 2 - motion_size // 2
        fixed_col = image_size // 2 - motion_size // 2

        for t in range(num_frames):
            if label == 0:
                row = fixed_row
                col = base + t * step
            else:
                row = base + t * step
                col = fixed_col
            video[t, :, row:row + motion_size, col:col + motion_size] += 1.5

        return video.clamp(-1.0, 2.0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]

train_dataset = SyntheticMotionVideoDataset(num_samples=400, image_size=IMAGE_SIZE, num_frames=NUM_FRAMES, motion_size=MOTION_SIZE)
val_dataset = SyntheticMotionVideoDataset(num_samples=120, image_size=IMAGE_SIZE, num_frames=NUM_FRAMES, motion_size=MOTION_SIZE)

print(f'Train set: {len(train_dataset)} videos')
print(f'Val set:   {len(val_dataset)} videos')
print(f'Video shape: {tuple(train_dataset[0][0].shape)}')  # (T, C, H, W)

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

sample_videos, sample_labels = next(iter(train_loader))
print(f'Batch video shape: {tuple(sample_videos.shape)}')   # (B, T, C, H, W)
print(f'Batch label shape: {tuple(sample_labels.shape)}')
print('Label names:', [CLASS_NAMES[i] for i in sample_labels[:8].tolist()])

fig, axes = plt.subplots(2, NUM_FRAMES, figsize=(10, 4))
for row, class_id in enumerate([0, 1]):
    video, _ = train_dataset[class_id]
    for t in range(NUM_FRAMES):
        frame = video[t].permute(1, 2, 0).numpy()
        frame = (frame - frame.min()) / (frame.max() - frame.min() + 1e-6)
        axes[row, t].imshow(frame)
        axes[row, t].set_title(f'class={CLASS_NAMES[class_id]}, t={t}')
        axes[row, t].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 128
DROPOUT = 0.1
LEARNING_RATE = 3e-4
LEARNING_EPOCHS = 6
ENGINEERING_EPOCHS = 6

NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
TOTAL_TOKENS = NUM_FRAMES * NUM_PATCHES + 1

print({
    'D_MODEL': D_MODEL,
    'NUM_HEADS': NUM_HEADS,
    'NUM_LAYERS': NUM_LAYERS,
    'D_FF': D_FF,
    'NUM_PATCHES_PER_FRAME': NUM_PATCHES,
    'TOTAL_TOKENS_WITH_CLS': TOTAL_TOKENS,
})

@dataclass
class TrainResult:
    losses: list
    accuracies: list


def accuracy_from_logits(logits, labels):
    return (logits.argmax(dim=-1) == labels).float().mean().item()


def run_epoch(model, loader, optimizer=None, forward_fn=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_acc, total_count = 0.0, 0.0, 0
    criterion = nn.CrossEntropyLoss()

    for videos, labels in loader:
        videos = videos.to(device)
        labels = labels.to(device)

        logits = forward_fn(model, videos)
        loss = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_acc += accuracy_from_logits(logits, labels) * batch_size
        total_count += batch_size

    return total_loss / total_count, total_acc / total_count


def train_classifier(model, train_loader, val_loader, epochs, lr, forward_fn):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = TrainResult(losses=[], accuracies=[])

    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer=optimizer, forward_fn=forward_fn)
        val_loss, val_acc = run_epoch(model, val_loader, optimizer=None, forward_fn=forward_fn)
        history.losses.append((train_loss, val_loss))
        history.accuracies.append((train_acc, val_acc))
        print(f'Epoch {epoch + 1:02d}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | train_acc={train_acc:.2%} | val_acc={val_acc:.2%}')

    return history, model


@torch.no_grad()
def predict_batch(model, videos, forward_fn):
    model.eval()
    videos = videos.to(device)
    logits = forward_fn(model, videos)
    probs = logits.softmax(dim=-1)
    preds = probs.argmax(dim=-1)
    return probs.cpu(), preds.cpu()

## Section iii：学习路径（L1：从零手写 tiny TimeSformer）

这条路径的目标不是追求 Kinetics 级别精度，而是把 TimeSformer 的数据流完整跑通：

`video -> patch embedding -> temporal attention -> spatial attention -> FFN -> CLS -> classifier`

下面按论文的数据流顺序，一个组件一个组件搭起来。每一步都尽量保留 shape 注释。

### 组件 1：Video Patch Embedding

输入视频 shape 为 `(batch, frames, channels, height, width)`。

每帧做 2D patch 切分后得到：
- 每帧 patch 数：$N=(H/P)\times(W/P)$
- 总 patch token 数：$T\times N$

公式可以写成：

$$x_{patch} = \mathrm{Proj}(\mathrm{Patchify}(video))$$

然后加上 `CLS token` 与位置编码：

$$z_0 = [x_{cls}; x_{patch}] + E_{pos}$$

shape 流：
- `(B, T, C, H, W)`
- `-> (B*T, C, H, W)`
- `-> (B*T, D, H/P, W/P)`
- `-> (B, T*N, D)`
- `-> (B, T*N+1, D)`

In [ ]:
class VideoPatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, num_frames, d_model):
        super().__init__()
        self.num_frames = num_frames
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.randn(1, num_frames * self.num_patches + 1, d_model) * 0.02)

    def forward(self, videos):
        # videos: (B, T, C, H, W)
        b, t, c, h, w = videos.shape
        x = videos.reshape(b * t, c, h, w)                 # (B*T, C, H, W)
        x = self.proj(x)                                   # (B*T, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)                   # (B*T, N, D)
        x = x.reshape(b, t * self.num_patches, -1)         # (B, T*N, D)
        cls = self.cls_token.expand(b, -1, -1)             # (B, 1, D)
        x = torch.cat([cls, x], dim=1)                     # (B, T*N+1, D)
        x = x + self.pos_embed                             # (B, T*N+1, D)
        return x

patch_embed = VideoPatchEmbedding(IMAGE_SIZE, PATCH_SIZE, NUM_FRAMES, D_MODEL)
embedded = patch_embed(sample_videos[:2])
print('Patch embedding output:', tuple(embedded.shape))

### 组件 2：Temporal Self-Attention

TimeSformer 的第一步注意力不是在整段视频上直接全连，而是：

**固定空间位置，只在时间维做注意力。**

也就是把 shape 从 `(B, T, N, D)` 变换成 `(B*N, T, D)`，让“同一个 patch 位置在不同帧上的 token”互相看到彼此。

公式：

$$\mathrm{TemporalAttn}(Q,K,V)=\mathrm{softmax}(QK^T / \sqrt{d_k})V$$

这里只是把注意力的作用范围限制在时间轴上。

In [ ]:
class TemporalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, num_frames, num_patches):
        # x: (B, T*N+1, D)
        b, _, d = x.shape
        cls_token = x[:, :1]                                              # (B, 1, D)
        patches = x[:, 1:].reshape(b, num_frames, num_patches, d)         # (B, T, N, D)
        patches = patches.permute(0, 2, 1, 3).reshape(b * num_patches, num_frames, d)  # (B*N, T, D)

        qkv = self.qkv(patches).reshape(b * num_patches, num_frames, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)                                  # (3, B*N, H, T, Dh)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale                     # (B*N, H, T, T)
        attn = self.dropout(attn.softmax(dim=-1))
        out = attn @ v                                                    # (B*N, H, T, Dh)
        out = out.transpose(1, 2).reshape(b * num_patches, num_frames, d) # (B*N, T, D)
        out = self.proj(out)

        out = out.reshape(b, num_patches, num_frames, d).permute(0, 2, 1, 3).reshape(b, num_frames * num_patches, d)
        return torch.cat([cls_token, out], dim=1)                         # (B, T*N+1, D)

temporal_attn = TemporalSelfAttention(D_MODEL, NUM_HEADS)
temporal_out = temporal_attn(embedded, NUM_FRAMES, NUM_PATCHES)
print('Temporal attention output:', tuple(temporal_out.shape))

### 组件 3：Spatial Self-Attention

第二步是 **固定帧，在帧内做空间注意力**。

这一步把 `(B, T, N, D)` 变换成 `(B*T, N+1, D)`，其中 `CLS token` 会被复制到每一帧前面，与该帧所有 patch 交互。

这样做的直觉是：
- temporal attention 学“同位置跨帧如何变化”
- spatial attention 学“同一帧内部各 patch 如何组成动作语义”

这就是论文的 **Divided Space-Time**。

In [ ]:
class SpatialSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, num_frames, num_patches):
        # x: (B, T*N+1, D)
        b, _, d = x.shape
        cls_token = x[:, :1]                                              # (B, 1, D)
        patches = x[:, 1:].reshape(b, num_frames, num_patches, d)         # (B, T, N, D)
        cls_each_frame = cls_token.unsqueeze(1).expand(-1, num_frames, -1, -1)  # (B, T, 1, D)
        frame_tokens = torch.cat([cls_each_frame, patches], dim=2)        # (B, T, N+1, D)
        frame_tokens = frame_tokens.reshape(b * num_frames, num_patches + 1, d)  # (B*T, N+1, D)

        qkv = self.qkv(frame_tokens).reshape(b * num_frames, num_patches + 1, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)                                  # (3, B*T, H, N+1, Dh)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale                     # (B*T, H, N+1, N+1)
        attn = self.dropout(attn.softmax(dim=-1))
        out = attn @ v                                                    # (B*T, H, N+1, Dh)
        out = out.transpose(1, 2).reshape(b * num_frames, num_patches + 1, d)
        out = self.proj(out)

        cls_out = out[:, :1].reshape(b, num_frames, 1, d).mean(dim=1)     # (B, 1, D)
        patch_out = out[:, 1:].reshape(b, num_frames * num_patches, d)    # (B, T*N, D)
        return torch.cat([cls_out, patch_out], dim=1)                     # (B, T*N+1, D)

spatial_attn = SpatialSelfAttention(D_MODEL, NUM_HEADS)
spatial_out = spatial_attn(temporal_out, NUM_FRAMES, NUM_PATCHES)
print('Spatial attention output:', tuple(spatial_out.shape))

### 组件 4：Divided Space-Time Block

完整 block 顺序：

$$x' = x + \mathrm{TemporalAttn}(\mathrm{LN}(x))$$
$$x'' = x' + \mathrm{SpatialAttn}(\mathrm{LN}(x'))$$
$$x_{out} = x'' + \mathrm{MLP}(\mathrm{LN}(x''))$$

和标准 Transformer block 相比，区别在于这里显式拆成了 **两段注意力**。

### 训练 vs 推理的区别
TimeSformer 做视频分类时，训练和推理的结构差异其实很小：
- **训练**：`model.train()`，启用 dropout，用带标签的交叉熵优化分类头。
- **推理**：`model.eval()`，关闭 dropout，只做前向得到 logits。

也就是说，它不像 GPT/seq2seq 那样存在 teacher forcing vs autoregressive 的显式分离；但 `train/eval` 模式切换仍然是必要的。

In [ ]:
class DividedSTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.temporal_attn = TemporalSelfAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.spatial_attn = SpatialSelfAttention(d_model, num_heads, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, num_frames, num_patches):
        x = x + self.temporal_attn(self.norm1(x), num_frames, num_patches)  # (B, T*N+1, D)
        x = x + self.spatial_attn(self.norm2(x), num_frames, num_patches)   # (B, T*N+1, D)
        x = x + self.mlp(self.norm3(x))                                      # (B, T*N+1, D)
        return x

block = DividedSTBlock(D_MODEL, NUM_HEADS, D_FF)
block_out = block(embedded, NUM_FRAMES, NUM_PATCHES)
print('Block output:', tuple(block_out.shape))

### 组件 5：完整 tiny TimeSformer

现在把 patch embedding、多个 divided block、分类头组装为一个最小可训练模型。

最终分类方式：

$$y = W \cdot \mathrm{LN}(x_{cls}^{(L)}) + b$$

其中 $x_{cls}^{(L)}$ 是最后一层 block 输出的 `CLS token`。

In [ ]:
class TinyTimeSformer(nn.Module):
    def __init__(self, image_size, patch_size, num_frames, d_model, num_heads, num_layers, d_ff, num_classes, dropout=0.1):
        super().__init__()
        self.num_frames = num_frames
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_embed = VideoPatchEmbedding(image_size, patch_size, num_frames, d_model)
        self.blocks = nn.ModuleList([
            DividedSTBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, videos):
        # videos: (B, T, C, H, W)
        x = self.patch_embed(videos)                                       # (B, T*N+1, D)
        for block in self.blocks:
            x = block(x, self.num_frames, self.num_patches)                # (B, T*N+1, D)
        cls_token = self.norm(x[:, 0])                                     # (B, D)
        logits = self.head(cls_token)                                      # (B, num_classes)
        return logits

learning_model = TinyTimeSformer(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    num_frames=NUM_FRAMES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
)

learning_logits = learning_model(sample_videos[:2])
print('Learning model output:', tuple(learning_logits.shape))
print('Learning params:', sum(p.numel() for p in learning_model.parameters()))

### 学习路径训练

下面正式训练手写版 tiny TimeSformer。

这一部分对应论文中的 supervised video classification 流程：
- 输入整段视频 clip
- 输出类别 logits
- 用交叉熵优化分类头和 backbone

In [ ]:
learning_history, learning_model = train_classifier(
    model=learning_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=LEARNING_EPOCHS,
    lr=LEARNING_RATE,
    forward_fn=lambda model, videos: model(videos),
)

### 学习路径推理

这里显式切换到 `eval()` 模式，关闭 dropout，只做前向推理。

虽然视频分类没有自回归解码，但这仍然是训练 / 推理区别最直接的代码位置。

In [ ]:
learning_probs, learning_preds = predict_batch(
    learning_model,
    sample_videos[:8],
    forward_fn=lambda model, videos: model(videos),
)

for idx in range(8):
    print(f'sample {idx}: true={CLASS_NAMES[sample_labels[idx].item()]:>10s} | pred={CLASS_NAMES[learning_preds[idx].item()]:>10s} | prob={learning_probs[idx].tolist()}')

## Section iv：工程路径（E1：Hugging Face TimeSformer）

这里使用 `transformers` 的 `TimesformerForVideoClassification`。

根据官方文档，TimeSformer 属于成熟的工程库实现：
- 提供 `TimesformerConfig`
- 提供 `TimesformerForVideoClassification`
- `pixel_values` 使用视频输入张量

为了和学习路径保持同任务、同数据、同训练目标，这里不直接加载大规模预训练权重，而是：
- 用 tiny config 构造一个小模型
- 仍然训练在同一个 toy dataset 上
- 让比较更公平、更稳定

### 学习路径实现 vs 工程库内部对应

| 学习路径（手写） | 工程路径（库内部） | 说明 |
|---|---|---|
| `VideoPatchEmbedding` | `TimesformerEmbeddings` | patch projection + CLS + position embeddings |
| `TemporalSelfAttention` | encoder block 内 temporal attention | 同一思想，库内部封装 |
| `SpatialSelfAttention` | encoder block 内 spatial attention | 同一思想，库内部封装 |
| `TinyTimeSformer.forward()` | `TimesformerForVideoClassification.forward()` | 输出分类 logits |
| 手写训练循环 | 同一个训练循环复用 | 工程化并不等于不训练，只是模块封装更高 |

In [ ]:
engineering_config = TimesformerConfig(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    num_frames=NUM_FRAMES,
    num_channels=3,
    hidden_size=D_MODEL,
    num_hidden_layers=NUM_LAYERS,
    num_attention_heads=NUM_HEADS,
    intermediate_size=D_FF,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    num_labels=NUM_CLASSES,
)

engineering_model = TimesformerForVideoClassification(engineering_config)
print('Engineering params:', engineering_model.num_parameters())


def engineering_forward(model, videos):
    # Hugging Face TimeSformer docs use pixel_values shaped as (batch, num_frames, channels, height, width)
    outputs = model(pixel_values=videos)
    return outputs.logits

engineering_logits = engineering_forward(engineering_model, sample_videos[:2])
print('Engineering model output:', tuple(engineering_logits.shape))

In [ ]:
engineering_history, engineering_model = train_classifier(
    model=engineering_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=ENGINEERING_EPOCHS,
    lr=LEARNING_RATE,
    forward_fn=engineering_forward,
)

engineering_probs, engineering_preds = predict_batch(
    engineering_model,
    sample_videos[:8],
    forward_fn=engineering_forward,
)

print('\nEngineering inference examples:')
for idx in range(8):
    print(f'sample {idx}: true={CLASS_NAMES[sample_labels[idx].item()]:>10s} | pred={CLASS_NAMES[engineering_preds[idx].item()]:>10s} | prob={engineering_probs[idx].tolist()}')

## Section v：学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 可以看到 patch、temporal、spatial、CLS 如何一步步流动 | 可以快速理解工业 API 如何封装论文结构 |
| 代码量 | 明显更多，需要手动处理 shape 与 residual | 明显更少，主要关注 config 与训练接口 |
| 灵活性 | 很高，任意组件都能改 | 中等，受限于库内部实现与暴露接口 |
| 稳定性 | 更容易写错 shape、residual、reshape | 更稳定，底层实现已被广泛复用 |
| 工业适配度 | 更适合教学、研究原型、论文复现 | 更适合实验管理、快速验证、工程落地 |
| 适用场景 | 面试准备；解释论文；尝试新注意力变体 | 快速搭 baseline；接入已有训练框架；复用预训练生态 |

In [ ]:
learning_train_losses = [x[0] for x in learning_history.losses]
learning_val_losses = [x[1] for x in learning_history.losses]
engineering_train_losses = [x[0] for x in engineering_history.losses]
engineering_val_losses = [x[1] for x in engineering_history.losses]

learning_val_accs = [x[1] for x in learning_history.accuracies]
engineering_val_accs = [x[1] for x in engineering_history.accuracies]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(learning_train_losses, label='Learning train')
axes[0].plot(learning_val_losses, label='Learning val')
axes[0].plot(engineering_train_losses, label='Engineering train')
axes[0].plot(engineering_val_losses, label='Engineering val')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(learning_val_accs, marker='o', label='Learning val acc')
axes[1].plot(engineering_val_accs, marker='s', label='Engineering val acc')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Final learning val acc:', f'{learning_val_accs[-1]:.2%}')
print('Final engineering val acc:', f'{engineering_val_accs[-1]:.2%}')

## Section vi：训练 vs 推理的区别

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `model.train()`，启用 dropout，手写 tiny TimeSformer 前向后接交叉熵 | `model.train()`，库内部 TimeSformer block 前向后接分类头与交叉熵 |
| 推理 | `model.eval()`，直接输出 logits，再 softmax 得到类别概率 | `model.eval()`，调用 `TimesformerForVideoClassification` 得到 `outputs.logits` |
| 输入形式 | `(batch, frames, channels, height, width)` | `(batch, frames, channels, height, width)` |
| 核心差别 | 结构透明、所有 shape 可见 | 结构封装、配置驱动 |

> 这类视频分类模型没有像 GPT 那样的自回归推理循环；训练 / 推理的核心区别主要在于 `train()` / `eval()` 模式以及是否反向传播。 

## Section vii：结果边界与实践建议

### 1. 学习路径的边界
- **得到的东西**：你能真正看懂 TimeSformer 的 token 重排、两段注意力、CLS 聚合方式。
- **失去的东西**：代码更长，更容易出 shape bug，也不具备大规模工程优化。

### 2. 工程路径的边界
- **得到的东西**：实现稳定、参数集中、接口统一，更适合接到现有训练系统里。
- **失去的东西**：很多关键细节被封装，看论文时不容易直接映射到每一行实现。

### 3. 实际怎么选
- **准备面试 / 精读论文**：先走学习路径。
- **做 baseline / 工程验证**：优先走工程路径。
- **要改论文结构**：先在学习路径验证，再迁移到工程实现。 

In [ ]:
fig, axes = plt.subplots(2, NUM_FRAMES, figsize=(10, 4))
for row, model_name in enumerate(['Learning', 'Engineering']):
    preds = learning_preds if model_name == 'Learning' else engineering_preds
    for t in range(NUM_FRAMES):
        frame = sample_videos[row][t].permute(1, 2, 0).numpy()
        frame = (frame - frame.min()) / (frame.max() - frame.min() + 1e-6)
        axes[row, t].imshow(frame)
        axes[row, t].set_title(f'{model_name} | t={t} | pred={CLASS_NAMES[preds[row].item()]}')
        axes[row, t].axis('off')
plt.tight_layout()
plt.show()

## Appendix B：面试与项目表达

### 高频面试题

**Q1：TimeSformer 为什么不直接做联合时空注意力，而要拆成 Temporal + Spatial？**

- 因为联合时空注意力的 token 数是 `T × N`，显存和计算都会快速膨胀。
- 拆分后每一步分别只关注时间轴或空间轴，复杂度更可控。
- 论文实验表明这种拆分不只是更省，还能得到更好的分类效果。

**Q2：Temporal attention 在看什么？Spatial attention 又在看什么？**

- Temporal attention：同一空间位置在不同帧中的变化。
- Spatial attention：同一帧内不同 patch 之间的空间关系。
- 两者组合起来，才形成完整的视频动作理解。

**Q3：TimeSformer 和 ViT 的核心差别是什么？**

- ViT 主要针对单张图像。
- TimeSformer 要处理视频，所以必须引入时间维。
- 它的关键改动不是把 2D patch 变成 3D patch，而是在 Transformer block 中显式引入时空注意力分解。

**Q4：为什么这份 toy dataset 能验证 TimeSformer 的价值？**

- 因为类别差异来自运动方向，而不是静态纹理。
- 如果模型只会看单帧，区分会更困难。
- 能把水平/垂直运动学出来，说明它确实利用了时间信息。

**Q5：手写实现和 Hugging Face 实现，最大的差别是什么？**

- 手写实现强调透明性，便于讲解原理。
- Hugging Face 实现强调稳定性和模块复用，便于快速接实验。
- 两者表达的是同一模型思想，只是抽象层不同。

**Q6：TimeSformer 相比 3D CNN 的优势是什么？**

- 更擅长建模长距离依赖。
- 不依赖卷积的局部归纳偏置，可以直接做全局关系建模。
- 但在小数据场景下，3D CNN 的强归纳偏置有时仍然有优势。

**Q7：如果视频更长、分辨率更高，TimeSformer 会遇到什么问题？**

- token 数会快速增长，注意力成本上升。
- 通常需要采样、更高效的注意力设计、层次化结构或 token 压缩。
- 这也是后续 Video Swin、MViT、VideoMAE 等方法继续优化的方向。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | TimeSformer 的核心创新是什么？ | 把视频注意力拆成 temporal + spatial 两步做。 |
| 2 | 为什么拆分注意力更好？ | 复杂度更可控，同时保留时空建模能力。 |
| 3 | Temporal attention 学什么？ | 学同一位置跨帧的变化规律。 |
| 4 | Spatial attention 学什么？ | 学同一帧内部 patch 的空间关系。 |
| 5 | Hugging Face 版本有什么价值？ | 便于快速实验、配置管理和工程接入。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我从零实现了 tiny TimeSformer，把 patch embedding、temporal attention、spatial attention 和 divided block 全部手写了一遍。
- **工程能力**：我又用 Hugging Face 的 `TimesformerForVideoClassification` 在同一任务上复现了一遍，验证了论文结构和工业 API 的映射关系。
- **对比思考**：我能清楚解释手写实现与工程库实现的差别：前者更透明，后者更稳定，适用场景不同。

### 延伸阅读与对比

| 对比维度 | TimeSformer | ViViT | Video Swin |
|---|---|---|---|
| 核心思想 | 拆分时空注意力 | 多种 factorized video transformer 设计 | 局部窗口时空注意力 |
| 全局建模 | 强 | 强 | 较弱但更高效 |
| 复杂度 | 中等偏高 | 依变体而定 | 更适合高分辨率 |
| 适用场景 | 论文讲解、全局视频理解 | 通用 Video Transformer 对照 | 工程效率优先 |

### 进阶探索方向

- 把 toy dataset 换成真实小视频数据集，例如 UCF101 子集。
- 把手写 temporal/spatial attention 改成 local attention，观察效率变化。
- 对比 TimeSformer、ViViT、Video Swin 在同一数据上的收敛曲线。 

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
complexity_names = ['S', 'ST', 'T+S', 'L+G', 'T+W+H']
complexity_scores = [NUM_PATCHES, NUM_PATCHES * NUM_FRAMES, NUM_PATCHES + NUM_FRAMES, int(math.sqrt(NUM_PATCHES * NUM_FRAMES) * 4), NUM_FRAMES + 2 * int(math.sqrt(NUM_PATCHES))]
ax.bar(complexity_names, complexity_scores, color=['#8ecae6', '#e63946', '#2a9d8f', '#f4a261', '#a78bfa'])
ax.set_title('Relative Attention Scope Cost')
ax.set_xlabel('Attention Variant')
ax.set_ylabel('Relative Cost')
for i, v in enumerate(complexity_scores):
    ax.text(i, v + 0.5, str(v), ha='center')
plt.tight_layout()
plt.show()

## Appendix A：注意力范围与复杂度补充

论文里对比了 5 种时空注意力思路：

| 方案 | 缩写 | 说明 | 复杂度直觉 |
|---|---|---|---|
| Space-only | S | 每帧单独看空间，不看跨帧关系 | 与单帧 ViT 类似 |
| Joint Space-Time | ST | 所有时空 token 一次性全连接 | 最贵 |
| Divided Space-Time | T+S | 先时间后空间 | 更均衡 |
| Local-Global | L+G | 先局部再全局稀疏 | 折中 |
| Axial | T+W+H | 沿不同轴分别做注意力 | 分解型 |

这份 Notebook 选择实现 **T+S**，因为它正是论文中最关键、也最有教学价值的方案。